# Glossolalia Dial — Micro-Spike

Glossolalia Dial — fine-tune an F5-TTS LoRA so a single control token (`tongues zero..four`) grades the intelligibility of the same input sentence in the same voice, from clean speech to phonotactically-valid English-native glossolalia. The fine-tune is the load-bearing trick: no prompt, no DSP plugin, and no shipped product (closed or open) does this — verified by three adversarial research workflows (originality landscape, Cocteau Twins frame audit, glossolalia-converter prior-art sweep).

**Gates that must ALL pass before we ship:**
- Whisper-WER rises monotonically across levels 0..4, Spearman ≥ +0.80 (positive sign — WER *rises* as dial rises).
- Resemblyzer cosine vs the level-0 reference stays ≥ 0.85 (voice preserved).
- Perceptual gate: hand-listen `dial=2` wavs — must be a partial dissolution, not bimodal collapse.

**Recommended order:** run the **200-clip micro-spike** (Cell 3 default, ~17 min on A100) first to confirm the LoRA learns the control token at all. If the micro-spike's Cell-7 verdict is PARTIAL or PASS, scale to the full 7,500-clip run before committing to the long training.

Runtime: A100/H100 ZeroGPU strongly recommended (F5-TTS ~5-7s per generated clip).

In [ ]:
# Cell 1 — install F5-TTS + validation deps + repo scripts
!nvidia-smi -L
!apt-get -qq install -y ffmpeg >/dev/null
!pip install -q f5-tts g2p_en jiwer openai-whisper resemblyzer huggingface_hub pedalboard soundfile librosa scipy numpy peft >/dev/null
!pip install -q -U "torchao>=0.16.0" >/dev/null  # peft >=0.14 requires torchao>=0.16 to import; Colab ships 0.10
import nltk
for res in ('averaged_perceptron_tagger_eng', 'averaged_perceptron_tagger', 'cmudict'):
    try: nltk.download(res, quiet=True)
    except Exception: pass
# Bring in the repo scripts. If you renamed the repo, adjust REPO_URL.
REPO_URL = 'https://github.com/akshan-main/glossolalia.git'
import os
if not os.path.isdir('/content/repo'):
    !git clone -q $REPO_URL /content/repo || echo 'clone failed - upload scripts/ manually via Files panel'
%cd /content/repo
!ls scripts/ | head -20
print('install done')

In [ ]:
# Cell 2 — pull data inputs (sentences + voice refs + phoneme LM) from HF
from huggingface_hub import snapshot_download
import shutil, os
REPO = 'akshan-main/glossolalia-inputs'   # populated by scripts/push_data_inputs.py before running this notebook
p = snapshot_download(repo_id=REPO, repo_type='dataset', local_dir='/content/repo/data_pull')
# Stage into the layout the scripts expect (data/sentences.txt, data/voices/v*.wav, data/phoneme_lm.npz)
os.makedirs('/content/repo/data', exist_ok=True)
os.makedirs('/content/repo/data/voices', exist_ok=True)
for f in ['sentences.txt', 'phoneme_lm.npz', 'cmudict.dict']:
    src = os.path.join(p, f)
    if os.path.exists(src): shutil.copy(src, f'/content/repo/data/{f}')
voices_dir = os.path.join(p, 'voices')
if os.path.isdir(voices_dir):
    for f in os.listdir(voices_dir):
        shutil.copy(os.path.join(voices_dir, f), f'/content/repo/data/voices/{f}')
print('  sentences:', len(open('/content/repo/data/sentences.txt').readlines()))
print('  voices   :', sorted(os.listdir('/content/repo/data/voices')))
assert os.path.exists('/content/repo/data/phoneme_lm.npz'), 'phoneme_lm.npz missing'

In [ ]:
# Cell 3 — generate the training corpus on GPU (F5-TTS base, no LoRA yet)
# Scales:
#   micro-spike   ->  40 sentences  x 5 levels x 1 voice =   200 clips  (~17 min on A100)
#   single-voice  -> 500 sentences  x 5 levels x 1 voice = 2,500 clips  (~3-4 h)
#   full          -> 500 sentences  x 5 levels x 3 voices = 7,500 clips (~10-12 h)
# Recommended: run the 200-clip micro-spike FIRST. Train a tiny LoRA on it (Cell 5 with --epochs 1)
# and read Cell 7's verdict. If Spearman > 0.5, scale up. If 0, the corruption procedure or token
# format needs fixing BEFORE you spend 10 hours.
!python scripts/generate_coherence_data.py \
  --sentences data/sentences.txt \
  --voice v1:data/voices/v1.wav:data/voices/v1.txt \
  --lm data/phoneme_lm.npz \
  --out data/coherence \
  --max-sentences 40 \
  --levels 5 \
  --input-mode pseudo \
  --resume
# To scale: add `--voice v2:data/voices/v2.wav:data/voices/v2.txt --voice v3:data/voices/v3.wav:data/voices/v3.txt`
# and bump `--max-sentences 500`.

In [ ]:
# Cell 4 — build the F5-TTS finetune dataset (CSV + JSONL + symlinked wavs)
!python scripts/build_coherence_dataset.py --data data/coherence --out data/coherence_ds
!head -3 data/coherence_ds/metadata.csv
!wc -l data/coherence_ds/metadata.csv

In [ ]:
# Cell 5 — LoRA finetune on F5-TTS (DIY PEFT path)
#
# Why DIY instead of a community fork (see DECISIONS.md "F5-TTS LoRA path = DIY PEFT"):
#  - Official f5-tts pip 1.1.20 has NO --lora flags; calling --lora_rank crashes
#  - instavar/f5-tts-lora-finetuning defaults --tokenizer pinyin (silent mistrain on English)
#    AND adds inference via __init__(lora_path=) not tts.load_lora(path) — our app.py expects
#    the method, not the kwarg
#  - PR #1296 on SWivid is unmerged + GPU-unvalidated + ships no inference loader
#
# This cell: build CFM(DiT) like official trainer.py, load base EMA, PEFT-wrap cfm.transformer
# (NOT cfm itself — that would clobber forward(mel,text=,lens=) signature), train via CFM.forward
# which returns flow-matching loss directly, save portable PEFT adapter dir, install monkey-patch
# so F5TTS.load_lora(path) works at inference.

EPOCHS = 1   # micro-spike. Bump to 5 for the full 7,500-clip run.
BATCH = 4
LR = 2e-4
OUT_DIR = '/content/repo/ckpts/unlang_lora'
ADAPTER_DIR = f'{OUT_DIR}/lora_last'

import os, time, csv
os.makedirs(OUT_DIR, exist_ok=True)

import torch, torchaudio
from torch.utils.data import Dataset, DataLoader
from cached_path import cached_path
from peft import LoraConfig, get_peft_model

from f5_tts.model import CFM, DiT
from f5_tts.model.utils import get_tokenizer
from f5_tts.infer.utils_infer import load_checkpoint

device = 'cuda' if torch.cuda.is_available() else 'cpu'
assert device == 'cuda', 'A100/H100 expected for the spike'

# 1. Build CFM(DiT) exactly like finetune_cli.py does for F5TTS_v1_Base
vocab_path = str(cached_path('hf://SWivid/F5-TTS/F5TTS_v1_Base/vocab.txt'))
vocab_char_map, vocab_size = get_tokenizer(vocab_path, 'custom')
mel_kwargs = dict(n_fft=1024, hop_length=256, win_length=1024,
                  n_mel_channels=100, target_sample_rate=24000, mel_spec_type='vocos')
dit = DiT(dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4,
          text_num_embeds=vocab_size, mel_dim=100, checkpoint_activations=True)
cfm = CFM(transformer=dit, mel_spec_kwargs=mel_kwargs,
          vocab_char_map=vocab_char_map).to(device)

# 2. Load base EMA weights
ckpt = str(cached_path('hf://SWivid/F5-TTS/F5TTS_v1_Base/model_1250000.safetensors'))
cfm = load_checkpoint(cfm, ckpt, device=device, use_ema=True)

# 3. Freeze base, PEFT-wrap the DiT (not the whole CFM)
for p in cfm.parameters(): p.requires_grad_(False)
lora_cfg = LoraConfig(r=16, lora_alpha=16, lora_dropout=0.0, bias='none',
                      target_modules=['to_q','to_k','to_v','to_out.0'])
cfm.transformer = get_peft_model(cfm.transformer, lora_cfg)
cfm.transformer.print_trainable_parameters()
# Sanity: 22 blocks * 4 matched linears = 88. Fail loud if PEFT version drifts the suffix match.
n_targets = sum(1 for n,_ in cfm.transformer.named_modules()
                if any(n.endswith('.'+t) and 'lora_' not in n for t in
                       ['to_q','to_k','to_v','to_out.0']))
print(f'  matched target Linears: {n_targets} (expect 88)')
assert n_targets == 88, 'target_modules suffix match is wrong — abort before wasting GPU'

# 4. Dataset over data/coherence_ds/metadata.csv
# CSV is `audio_path|text` — but text contains its own `|` (sentence | tongues N control token),
# so DON'T use csv.reader. Split on first | only. Duration not in CSV; read it from wav header.
from pathlib import Path as _Path
class CoherenceDS(Dataset):
    def __init__(self, csv_path, mel_kwargs, max_dur=15.0):
        self.rows = []
        base = _Path(csv_path).parent
        with open(csv_path) as f:
            f.readline()  # skip "audio_path|text" header
            for line in f:
                line = line.rstrip('\n')
                if not line: continue
                parts = line.split('|', 1)
                if len(parts) < 2: continue
                rel_path, text = parts[0].strip(), parts[1].strip()
                wav_p = base / rel_path
                try:
                    info = torchaudio.info(str(wav_p))
                    dur = info.num_frames / info.sample_rate
                    if dur > max_dur: continue
                except Exception:
                    continue
                self.rows.append((str(wav_p), text))
        from torchaudio.transforms import MelSpectrogram
        self.mel = MelSpectrogram(
            sample_rate=mel_kwargs['target_sample_rate'],
            n_fft=mel_kwargs['n_fft'], hop_length=mel_kwargs['hop_length'],
            win_length=mel_kwargs['win_length'], n_mels=mel_kwargs['n_mel_channels'],
            power=1, center=True, normalized=False)
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        wav_p, text = self.rows[i]
        y, sr = torchaudio.load(wav_p)
        if sr != 24000:
            y = torchaudio.functional.resample(y, sr, 24000)
        if y.shape[0] > 1: y = y.mean(0, keepdim=True)
        m = self.mel(y).squeeze(0).clamp(min=1e-5).log()
        return {'mel': m.transpose(0,1), 'text': text, 'mel_len': m.shape[-1]}

def collate(batch):
    maxT = max(b['mel'].shape[0] for b in batch)
    mels = torch.zeros(len(batch), maxT, 100)
    lens = torch.zeros(len(batch), dtype=torch.long)
    texts = []
    for i, b in enumerate(batch):
        T = b['mel'].shape[0]
        mels[i, :T] = b['mel']
        lens[i] = T
        texts.append(b['text'])
    return {'mel': mels, 'mel_lens': lens, 'text': texts}

ds = CoherenceDS('/content/repo/data/coherence_ds/metadata.csv', mel_kwargs)
dl = DataLoader(ds, batch_size=BATCH, shuffle=True, collate_fn=collate,
                num_workers=2, drop_last=True)
print(f'  dataset rows: {len(ds)}, batches/epoch: {len(dl)}')

# 5. Train loop — CFM.forward returns (loss, cond, pred)
trainable = [p for p in cfm.parameters() if p.requires_grad]
opt = torch.optim.AdamW(trainable, lr=LR, betas=(0.9, 0.99), weight_decay=0.0)
scaler = torch.amp.GradScaler('cuda')
warmup = 30
cfm.train()
step = 0; t0 = time.time()
for ep in range(EPOCHS):
    for batch in dl:
        mel = batch['mel'].to(device)
        mel_lens = batch['mel_lens'].to(device)
        text_in = batch['text']
        lr_now = LR * min(1.0, (step + 1) / warmup)
        for g in opt.param_groups: g['lr'] = lr_now
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            loss, _, _ = cfm(mel, text=text_in, lens=mel_lens)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(trainable, 1.0)
        scaler.step(opt); scaler.update()
        if step % 10 == 0:
            print(f'  ep{ep} step{step:4d} loss={loss.item():.4f} lr={lr_now:.2e} '
                  f'elapsed={time.time()-t0:.0f}s')
        step += 1

# 6. Save PEFT adapter dir
cfm.transformer.save_pretrained(ADAPTER_DIR, safe_serialization=True)
print(f'\nsaved adapter -> {ADAPTER_DIR}')
print('  files:', os.listdir(ADAPTER_DIR))

# 7. Activate the load_lora monkey-patch in this kernel so Cell 6 + later cells work
import sys
sys.path.insert(0, '/content/repo')
import patches  # installs F5TTS.load_lora
from f5_tts.api import F5TTS as _F5TTS_check
assert hasattr(_F5TTS_check, 'load_lora'), 'patch install failed'
print('F5TTS.load_lora installed:', hasattr(_F5TTS_check, 'load_lora'))
print('checkpoints:', [ADAPTER_DIR])


In [ ]:
# Cell 6 — sweep the dial across (sentence, voice, level, seed)
# Cell 5 writes a PEFT adapter directory (not a single .safetensors blob).
# Point sweep_dial.py at the dir. patches/ is auto-imported by sweep_dial.py itself.
import os, sys
sys.path.insert(0, '/content/repo')
import patches  # reinstall load_lora in case kernel was restarted
ADAPTER_DIR = '/content/repo/ckpts/unlang_lora/lora_last'
assert os.path.isfile(f'{ADAPTER_DIR}/adapter_model.safetensors'), 'Cell 5 must finish first'
print('using', ADAPTER_DIR)
!python scripts/sweep_dial.py \
  --lora $ADAPTER_DIR \
  --voices v1:data/voices/v1.wav:data/voices/v1.txt \
  --out sweep \
  --levels 5 \
  --seeds 42,123,7 \
  --max-sentences 10
# Multi-voice scale-up after micro-spike:
# --voices v1:data/voices/v1.wav:data/voices/v1.txt,v2:data/voices/v2.wav:data/voices/v2.txt,v3:data/voices/v3.wav:data/voices/v3.txt


In [ ]:
# Cell 7 — validate: Whisper WER (monotonic + Spearman ≥ +0.80) + Resemblyzer cosine (≥ 0.85)
!python scripts/evaluate_coherence_dial.py \
  --manifest sweep/sweep_manifest.json \
  --out sweep/eval_report.json \
  --whisper base.en \
  --spearman-gate 0.80 \
  --cosine-gate 0.85

In [ ]:
# Cell 8 — download dial=0/2/4 wavs for the human perceptual gate
from google.colab import files
import glob
picks = []
for lv in (0, 2, 4):
    g = sorted(glob.glob(f'sweep/v1_lv{lv}_s42_*.wav'))
    if g: picks.append(g[0])
for p in picks:
    print(p); files.download(p)

## Reading the result

- **All gates pass** (Spearman ≥ +0.80, min cosine ≥ 0.85, dial=2 perceptually partial) → push the LoRA + sweep wavs, build the toy.
- **Micro-spike Spearman 0** → the LoRA isn't learning the token. Check: is `tongues four` being split into chars (look at training logs)? Did F5-TTS swallow the `|` separator? Try the community LoRA fork.
- **Micro-spike Spearman 0.3-0.6** → real signal, just under-trained. Scale to full 7,500-clip run + 5 epochs.
- **WER non-monotonic** at full scale → tighten Markov LM to a vowel-heavy / sonorant-consonant palette and retrain.
- **Voice cosine < 0.85 at high levels** → lower rank to 8 + restrict target_modules to text-cross-attn only.
- **Dial=2 bimodal** → flatten the p-schedule (e.g., `[0, 0.20, 0.45, 0.70, 0.95]`) and curriculum-finetune on level-2-weighted mini-batches.

Push the trained LoRA:
```python
from huggingface_hub import HfApi
HfApi().upload_folder(folder_path='/content/repo/ckpts/unlang_lora', repo_id='akshan-main/glossolalia-dial-lora', repo_type='model')
```